# Lab 9: Joule A2A — Connect the Warehouse Agent into SAP Joule

In this lab, you will expose the warehouse operations agent (deployed to Amazon Bedrock AgentCore in Lab 8) as a **Joule Pro-code capability** so that SAP Joule can invoke it over the A2A protocol.

**Reference:** [Joule A2A – Connect Code-Based Agents into Joule (SAP Community Blog)](https://community.sap.com/t5/technology-blog-posts-by-sap/joule-a2a-connect-code-based-agents-into-joule/ba-p/14329279)

## What you will accomplish

1. Configure IAS credentials and log in with the Joule CLI
2. Author a Joule Pro-code capability (YAML files) that routes user questions to the AgentCore warehouse agent via A2A
3. Create a BTP Destination pointing to the AgentCore endpoint with OAuth2 client credentials
4. Deploy the capability with `joule deploy` and launch a test bot with `joule launch`
5. Observe a live Joule invocation of the warehouse agent

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                        SAP BTP Subaccount                       │
│                                                                 │
│   ┌─────────────────────────────────────────────────────────┐  │
│   │  SAP Joule                                              │  │
│   │   ├── AI Agent  (routes via scenario description)       │  │
│   │   └── A2A Client ──► capability.sapdas.yaml             │  │
│   └─────────────────────────┬───────────────────────────────┘  │
│                             │  A2A JSON-RPC 2.0 / HTTPS        │
│                             ▼                                   │
│       BTP Destination: warehouse-ops                            │
│       Authentication: OAuth2ClientCredentials (Cognito)         │
└─────────────────────────────┬───────────────────────────────────┘
                              │  HTTPS + Bearer token
                              ▼
          ┌──────────────────────────────────────────────┐
          │  Amazon Bedrock AgentCore Runtime             │
          │  Warehouse Operations Agent (from Lab 8)      │
          │  • Calls SAP S/4HANA OData via SAP GenAI Hub │
          └──────────────────────────────────────────────┘
```

**Key A2A concepts from the blog:**
- **Agent Card** (`/.well-known/agent.json`) — AgentCore exposes this; Joule auto-discovers the agent capabilities and endpoint from the Destination base URL
- **`agent-request` action** — the YAML action type that delegates a task to a remote A2A agent
- **`joule.ext` namespace** — mandatory for custom capabilities; any other namespace causes a deployment authorization error (error 5014)

---
## Section 1 — Setup: Install Joule CLI and Log In

### 1.1 Prerequisites

| Prerequisite | Details |
|---|---|
| SAP BTP Subaccount | Joule instance already subscribed |
| BTP roles assigned | `capability_admin`, `extensibility_developer`, `end_user` on the Joule application |
| SAP IAS tenant | Trusted by the BTP subaccount (found under **Trust Configuration**) |
| Node.js ≥ 18 | Required to install the Joule CLI via npm |

### 1.2 Install the Joule CLI

```bash
npm install -g @sap/joule-studio-cli
joule --version
```

### 1.3 Create IAS Application for CLI Authentication

Because Joule uses Cloud Identity Services (IAS) — not XSUAA — you must create a dedicated IAS application to obtain client credentials for the CLI:

1. Open the IAS Admin Panel: `https://<your-ias-tenant>.accounts.ondemand.com/admin`
2. Navigate to **Applications & Resources → Applications → Create**
   - Display Name: `Joule Pro Code CLI`
   - Protocol Type: **OpenID Connect**
3. Open the newly created application → **Dependencies → Add**
   - **Dependency Name: `Cli2Joule`** *(case-sensitive, must be exactly this)*
   - Application: select your Joule app (named `das-ias (<subaccount-name>)`)
   - API: `joule-dt-api`
4. Navigate to **Client Authentication → Add** a new secret
   - Tick: Application, Application Users, OpenID
   - Copy the **Client ID** and **Client Secret**

### 1.4 Log In with the Joule CLI

```bash
joule login
# ✔ Authentication URL: https://<tenant-id>.accounts.ondemand.com   # IAS domain (from admin URL)
# ✔ API URL:            https://<subdomain>.<region>.sapdas.cloud.sap  # Joule app URL from BTP
# ✔ Instance Client ID:     <client-id from step 1.3>
# ✔ Instance Client Secret: <client-secret from step 1.3>
# ✔ Username: your-email@company.com
# ✔ Password: ****
```

> **Authentication URL** = the IAS tenant domain (visible in the browser address bar of the admin panel).  
> **API URL** = the URL of your Joule application as shown in BTP Cockpit.

---
## Section 2 — Author the Joule Pro-Code Capability (YAML)

A Joule Pro-Code capability is a small folder of YAML files — no Node.js, no Docker, no `cf push`. You author the files locally and the Joule CLI compiles and deploys them.

### 2.1 Project Structure

```
warehouseagentcapability/           ← capability folder
├── capability.sapdas.yaml          ← capability descriptor + system alias
├── capability_context.yaml         ← shared context variables (contextId, taskId)
├── scenarios/
│   └── warehouseagentscenario.yaml ← intent description Joule uses for routing
└── functions/
    └── warehouseagentfunction.yaml ← action list: agent-request + message
da.sapdas.yaml                      ← assistant descriptor (for standalone test bot)
```

### 2.2 `capability.sapdas.yaml`

```yaml
schema_version: 3.27.0   # minimum version for agent-request support

metadata:
  namespace: joule.ext    # MUST be joule.ext — any other value → error 5014
  name: warehouse_ops_capability
  version: 1.0.0
  display_name: Warehouse Operations Capability
  description: Capability containing the Warehouse Operations agent

system_aliases:
  warehouse_ops:
    destination: warehouse-ops  # must match the BTP Destination name exactly
```

> **Critical:** Use `schema_version: 3.27.0` or higher — earlier versions do not support the `agent-request` action type.  
> **Critical:** Use `namespace: joule.ext` — SAP-owned namespaces are rejected.

### 2.3 `capability_context.yaml`

```yaml
variables:
  - name: contextId
  - name: taskId
```

Declares the capability-level context variables that persist across conversation turns. `contextId` and `taskId` are threaded from the agent response back into subsequent requests, enabling multi-turn dialogue.

### 2.4 `scenarios/warehouseagentscenario.yaml`

```yaml
description: The Warehouse Agent supports querying SAP supply chain system using natural language
target:
  name: warehouse_agent_function
  type: function
  parameters:
    - name: contextId
      value: $capability_context.contextId
    - name: taskId
      value: $capability_context.taskId

capability_context:
    - name: contextId
      value: $target_result.contextId
    - name: taskId
      value: $target_result.taskId
```

Joule evaluates this `description` during intent routing to decide which capability to invoke. The `capability_context` block enables multi-turn conversations by threading `contextId` and `taskId` across turns.

### 2.5 `functions/warehouseagentfunction.yaml`

```yaml
parameters:
  - name: contextId
    optional: true
  - name: taskId
    optional: true
action_groups:
  - actions:
      - type: status-update
        message: <? "Invoking Warehouse Operations Agent" ?>
      - type: agent-request
        agent_type: remote
        system_alias: warehouse_ops
        body: |
          { "contextId": <? contextId ?>, "taskId": <? taskId ?> }
        result_variable: result
      - type: set-variables
        scripting_type: handlebars
        variables:
          - name: concatenatedText
            value: '{{#if result.body.artifacts}}{{#each result.body.artifacts}}{{#if
              this.parts}}{{#each this.parts}}{{#if (and (eq kind "text")
              text)}}{{text}}{{/if}}{{/each}}{{/if}}{{#unless
              @first}}{{else}}{{/unless}}{{/each}}{{/if}}'
      - type: set-variables
        variables:
          - name: contextId
            value: <? result.body.contextId ?>
          - name: taskId
            value: <? result.body.id ?>
      - type: message
        message:
          type: text
          content: <? concatenatedText ?>
      - type: status-update
        message: <? "Completed Warehouse Operations Agent" ?>
result:
  contextId: <? contextId ?>
  taskId: <? taskId ?>
```

**Understanding `result_variable`:**  
The A2A response JSON has this shape:
```json
{
  "body": {
    "artifacts": [{ "parts": [{ "kind": "text", "text": "10 USD is 8.43 EUR..." }] }],
    "status": { "state": "completed" },
    "contextId": "...",
    "id": "..."
  }
}
```
The `set-variables` with Handlebars extracts text from all artifact parts. The `contextId` and `taskId` are threaded back via the `result` block for multi-turn support.

### 2.6 `da.sapdas.yaml` (for standalone test bot)

```yaml
schema_version: 1.2.0
name: warehouse_agent_test
capabilities:
  - type: local
    folder: ./warehouseagentcapability
```

This file is only needed when deploying a **standalone test assistant** with `joule deploy -n`. If you are adding the capability to the production `sap_digital_assistant`, you omit this file and use `joule update` instead.

> **Note — How the Handlebars `set-variables` block works**
>
> The A2A response can contain **multiple artifacts**, each with **multiple parts**. The Handlebars expression iterates over every artifact (`{{#each result.body.artifacts}}`) and every part within it (`{{#each this.parts}}`), keeping only parts where `kind` equals `"text"` and `text` is non-empty (`{{#if (and (eq kind "text") text)}}`). All matching text values are concatenated in order into the `concatenatedText` variable, which is then passed to the `message` action for display.
>
> This makes the extractor robust to multi-artifact, multi-part responses.

---
## Section 3 — Write the Capability Files

Run the cells below to create the capability folder and all required YAML files on disk, ready to deploy with the Joule CLI.

In [ ]:
import os, pathlib

# ── Create directory structure ────────────────────────────────────────────────
base = pathlib.Path("warehouseagentcapability")
(base / "scenarios").mkdir(parents=True, exist_ok=True)
(base / "functions").mkdir(parents=True, exist_ok=True)
print("✓ Directory structure created")

# ── capability.sapdas.yaml ────────────────────────────────────────────────────
(base / "capability.sapdas.yaml").write_text("""\
schema_version: 3.27.0

metadata:
  namespace: joule.ext
  name: warehouse_ops_capability
  version: 1.0.0
  display_name: Warehouse Operations Capability
  description: Capability containing the Warehouse Operations agent

system_aliases:
  warehouse_ops:
    destination: warehouse-ops
""")
print("✓ capability.sapdas.yaml written")

# ── capability_context.yaml ───────────────────────────────────────────────────
(base / "capability_context.yaml").write_text("""\
variables:
  - name: contextId
  - name: taskId
""")
print("✓ capability_context.yaml written")

# ── scenarios/warehouseagentscenario.yaml ─────────────────────────────────────
(base / "scenarios" / "warehouseagentscenario.yaml").write_text("""\
description: The Warehouse Agent supports querying SAP supply chain system using natural language
target:
  name: warehouse_agent_function
  type: function
  parameters:
    - name: contextId
      value: $capability_context.contextId
    - name: taskId
      value: $capability_context.taskId

capability_context:
    - name: contextId
      value: $target_result.contextId
    - name: taskId
      value: $target_result.taskId
""")
print("✓ scenarios/warehouseagentscenario.yaml written")

# ── functions/warehouseagentfunction.yaml ────────────────────────────────────
(base / "functions" / "warehouseagentfunction.yaml").write_text("""\
parameters:
  - name: contextId
    optional: true
  - name: taskId
    optional: true
action_groups:
  - actions:
      - type: status-update
        message: <? "Invoking Warehouse Operations Agent" ?>
      - type: agent-request
        agent_type: remote
        system_alias: warehouse_ops
        body: |
          { "contextId": <? contextId ?>, "taskId": <? taskId ?> }
        result_variable: result
      - type: set-variables
        scripting_type: handlebars
        variables:
          - name: concatenatedText
            value: '{{#if result.body.artifacts}}{{#each result.body.artifacts}}{{#if
              this.parts}}{{#each this.parts}}{{#if (and (eq kind "text")
              text)}}{{text}}{{/if}}{{/each}}{{/if}}{{#unless
              @first}}{{else}}{{/unless}}{{/each}}{{/if}}'
      - type: set-variables
        variables:
          - name: contextId
            value: <? result.body.contextId ?>
          - name: taskId
            value: <? result.body.id ?>
      - type: message
        message:
          type: text
          content: <? concatenatedText ?>
      - type: status-update
        message: <? "Completed Warehouse Operations Agent" ?>
result:
  contextId: <? contextId ?>
  taskId: <? taskId ?>
""")
print("✓ functions/warehouseagentfunction.yaml written")

# ── da.sapdas.yaml (top-level — for standalone test bot) ─────────────────────
pathlib.Path("da.sapdas.yaml").write_text("""\
schema_version: 1.2.0
name: warehouse_agent_test
capabilities:
  - type: local
    folder: ./warehouseagentcapability
""")
print("✓ da.sapdas.yaml written")

print("\nAll capability files created. Run the next cell to verify.")


In [ ]:
import subprocess
result = subprocess.run(["find", "warehouseagentcapability", "da.sapdas.yaml", "-type", "f"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else result.stderr)


---
## Section 4 — Create the BTP Destination

The Joule capability uses a BTP Destination to reach the AgentCore agent. When Joule triggers an `agent-request`, it:
1. Looks up the destination `warehouse-ops`
2. Fetches the Agent Card from `<destination-url>/.well-known/agent.json`
3. Sends A2A `message/send` JSON-RPC 2.0 requests to the URL declared in the Agent Card

### 4.1 Destination Configuration

In **BTP Cockpit → Connectivity → Destinations → New Destination**, create:

| Field | Value | Source |
|---|---|---|
| **Name** | `warehouse-ops` | Must match `system_aliases.warehouse_ops.destination` in `capability.sapdas.yaml` |
| **Type** | `HTTP` | Fixed |
| **URL** | `<Invoke URL from Lab 8 Step 12>` | Lab 8 output |
| **Proxy Type** | `Internet` | Fixed |
| **Authentication** | `OAuth2ClientCredentials` | Fixed |
| **Client ID** | `<Client ID from Lab 8 Step 12>` | Lab 8 output |
| **Client Secret** | `<Client Secret from Lab 8 Step 12>` | Lab 8 output |
| **Token Service URL** | `<Token Service URL from Lab 8 Step 12>` | Lab 8 output |

### 4.2 How Joule Discovers the Agent

Joule does **not** call the invoke URL directly. It:
1. Appends `/.well-known/agent.json` to the Destination URL → fetches the **Agent Card**
2. Reads the `url` field inside the Agent Card (the actual A2A endpoint)
3. Sends tasks to that URL with the OAuth2 bearer token from the Destination

AgentCore publishes a standards-compliant Agent Card automatically — no manual configuration needed.

### 4.3 Collect the Destination values from Lab 8

Run the next cell to print the values you need.

In [ ]:
import os
import subprocess
from urllib.parse import quote

# ── Read variables set by Lab 8 ───────────────────────────────────────────────
region         = os.getenv("REGION", "<not set — re-run Lab 8>")
client_id      = os.getenv("CLIENT_ID", "<not set — re-run Lab 8>")
cognito_domain = os.getenv("COGNITO_DOMAIN", "<not set — re-run Lab 8>")
oauth_scope    = os.getenv("OAUTH_SCOPE", "<not set — re-run Lab 8>")
user_pool_id   = os.getenv("USER_POOL_ID", "")

# Recover agent ARN from env or ~/.bedrock_agentcore.yaml
agent_arn = os.getenv("AGENT_ARN", "")
if not agent_arn:
    try:
        import yaml
        from pathlib import Path
        cfg = Path.home() / ".bedrock_agentcore.yaml"
        if cfg.exists():
            agent_arn = yaml.safe_load(cfg.read_text()).get("agent_arn", "")
    except Exception:
        pass

# Build the invoke URL
if agent_arn and "not set" not in region:
    encoded_arn = quote(agent_arn, safe="")
    invoke_url  = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
else:
    invoke_url = "<not available — re-run Lab 8>"

# Retrieve client secret from Cognito
client_secret = "<not available>"
if user_pool_id and client_id and "not set" not in client_id:
    r = subprocess.run(
        ["aws", "cognito-idp", "describe-user-pool-client",
         "--user-pool-id", user_pool_id, "--client-id", client_id,
         "--region", region, "--query", "UserPoolClient.ClientSecret", "--output", "text"],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        client_secret = r.stdout.strip()

token_service_url = (f"{cognito_domain}/oauth2/token"
                     if "not set" not in cognito_domain else cognito_domain)

# ── Print values to copy into BTP Cockpit ────────────────────────────────────
print("=" * 65)
print("  BTP DESTINATION CONFIGURATION  (copy into BTP Cockpit)")
print("=" * 65)
print(f"  Name             : warehouse-ops")
print(f"  Type             : HTTP")
print(f"  URL              : {invoke_url}")
print(f"  Proxy Type       : Internet")
print(f"  Authentication   : OAuth2ClientCredentials")
print(f"  Client ID        : {client_id}")
print(f"  Client Secret    : {client_secret}")
print(f"  Token Service URL: {token_service_url}")
print("=" * 65)


---
## Section 5 — Deploy the Joule Capability

The Joule CLI handles the full lifecycle in three phases:

1. **Compile** — packages the YAML files into a `.daar` archive (Design-time Artifact Archive)
2. **Deploy** — uploads the archive and registers the capability with a named assistant
3. **Launch** — returns a URL where you can open the test assistant in a browser

### 5.1 Deploy a Standalone Test Bot

Run this from the **top-level directory** (not inside the capability folder):

```bash
joule deploy -c -n "warehouse_agent_test"
```

- `-c` — compile first
- `-n "<bot_name>"` — create a standalone sandbox assistant for testing

**Expected output:**
```
✔ Building designtime artifact (warehouse_agent_capability)
✔ Trigger compilation (...)
✔ Compiled (...)
Detailed logs:
WARNINGS:
  Message: DTA did not define an optional i18n folder
  Category: I18N  Severity: LOW
✔ Downloaded runtime artifact (joule.ext_warehouse_agent_capability_1.0.0.daar)
✔ Building runtime artifact
✔ Triggering deployment (warehouse_agent_test)
✔ Your digital assistant (warehouse_agent_test) deployed successfully
```

### 5.2 Launch the Test Bot

```bash
joule launch "warehouse_agent_test"
# ✔ Launching: https://joule-pro-code-<id>.<region>.sapdas.cloud.sap/webclient/standalone/warehouse_agent_test
```

Open the URL and ask a warehouse question like _"What stock do we have for WM-AN02?"_

### 5.3 Update the Production Assistant (when ready)

Once validated, add the capability to the live Joule instance alongside SAP's standard content:

```bash
# Run from inside the capability folder
joule update "sap_digital_assistant" --capability-file capability.sapdas.yaml
```

> This pushes the capability into `sap_digital_assistant` (the assistant users reach via `/joule` in SAP S/4HANA) without creating a new bot.

---
## Section 6 — Verify: Direct Agent Call Before Joule Test

Before testing in Joule, confirm your AgentCore agent is still healthy by calling it directly with the JSON-RPC 2.0 payload. This is the same call Joule will make internally.

In [ ]:
import requests, uuid, json, os
from urllib.parse import quote

# ── Load variables (set by Lab 8 or Section 4 above) ─────────────────────────
bearer_token = os.getenv("BEARER_TOKEN", "")
region       = os.getenv("REGION", "")
agent_arn    = os.getenv("AGENT_ARN", "")

if not agent_arn:
    try:
        import yaml; from pathlib import Path
        cfg = Path.home() / ".bedrock_agentcore.yaml"
        if cfg.exists():
            agent_arn = yaml.safe_load(cfg.read_text()).get("agent_arn", "")
    except Exception:
        pass

if not invoke_url or "not available" in invoke_url:
    if agent_arn and region:
        invoke_url = (f"https://bedrock-agentcore.{region}.amazonaws.com"
                      f"/runtimes/{quote(agent_arn, safe='')}/invocations?qualifier=DEFAULT")

if not bearer_token:
    print("BEARER_TOKEN not set. Re-run Lab 8 Step 6c to obtain a fresh token.")
elif not invoke_url or "not available" in invoke_url:
    print("invoke_url not available. Re-run Section 4 in this notebook.")
else:
    query = "What stock do we have for WM-AN02?"
    payload = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "message/send",
        "params": {
            "message": {
                "role": "user",
                "parts": [{"kind": "text", "text": query}],
                "messageId": str(uuid.uuid4())
            }
        }
    }

    print(f"Query : {query}")
    print(f"URL   : {invoke_url}\n")

    response = requests.post(
        invoke_url,
        headers={"Authorization": f"Bearer {bearer_token}", "Content-Type": "application/json"},
        json=payload,
        timeout=60
    )

    print(f"HTTP  : {response.status_code}")
    if response.status_code == 200:
        rpc = response.json()
        parts  = rpc.get("result", {}).get("parts", [])
        answer = " ".join(p.get("text", "") for p in parts if p.get("kind") == "text")
        print(f"Answer: {answer if answer else json.dumps(rpc, indent=2)}")
    else:
        print(f"Error : {response.text}")


---
## Section 7 — End-to-End Flow: What to Observe in Joule

After deploying and launching the test bot, open the URL printed by `joule launch` and ask a warehouse question.

### What happens step by step

| Step | Component | Action |
|---|---|---|
| 1 | Joule AI Agent | Evaluates the user message against available scenario descriptions |
| 2 | Joule | Matches "warehouse stock / inventory" → selects `warehouseagentscenario` |
| 3 | Joule function executor | Runs `warehouseagentfunction.yaml` — fires the `agent-request` action |
| 4 | BTP Destination service | Exchanges Cognito client credentials → bearer token |
| 5 | Joule A2A client | Sends `POST /.well-known/agent.json` → discovers agent endpoint |
| 6 | Joule A2A client | Sends JSON-RPC 2.0 `message/send` to AgentCore invoke URL |
| 7 | AgentCore (warehouse agent) | Calls SAP S/4HANA OData via SAP GenAI Hub |
| 8 | AgentCore | Returns A2A response with `artifacts[0].parts[0].text` |
| 9 | Joule | Extracts text via `<? apiResponse.body.artifacts[0].parts[0].text ?>` |
| 10 | Joule chat UI | Displays the answer conversationally to the user |

### Sample questions to try

- _"What stock do we have for WM-AN02?"_
- _"Can we fulfill an order for 50 units of WM-AN01 sensors?"_
- _"What are the inventory levels for WM-AN03 and WM-AN04?"_

### Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `error 5014` on deploy | Wrong namespace | Use `namespace: joule.ext` in `capability.sapdas.yaml` |
| `AUTH_LOAD_FAILED` on login | Wrong Authentication URL | Use the IAS tenant domain, not the XSUAA URL |
| `invalid_client` / Bad credentials | Dependency name wrong | Ensure dependency is named exactly `Cli2Joule` (case-sensitive) |
| No scenario matched | Scenario description too generic | Make the description specific to warehouse/inventory queries |
| `Connection failed` in Destination test | Cognito domain not ready | Wait 1–2 minutes after domain creation and retry |

---
## Summary

In this lab you connected a custom AI agent running on AWS to SAP Joule using the open A2A protocol — with no changes to the agent code.

| Topic | Key Takeaway |
|---|---|
| Joule Pro-Code Capability | Three YAML files — no Node.js, no Docker, no `cf push` |
| `joule.ext` namespace | Mandatory for extensions; any SAP-owned namespace is rejected (error 5014) |
| `schema_version: 3.27.0` | Minimum version for the `agent-request` action type |
| `agent-request` action | The single YAML action that delegates a task to a remote A2A agent |
| Agent Card discovery | Joule auto-discovers the endpoint from `<destination-url>/.well-known/agent.json` |
| BTP Destination | Handles OAuth2 token exchange transparently — no credentials in YAML |
| `result_variable` | Dot-notation path `apiResponse.body.artifacts[0].parts[0].text` extracts the answer |
| `joule deploy -c -n` | Compile + deploy to a sandbox bot; use `joule update` for production |
| Cross-cloud A2A | An agent running on AWS AgentCore is invokable from SAP Joule with no agent code changes |

### Next Steps

1. Deploy to production: `joule update "sap_digital_assistant" --capability-file capability.sapdas.yaml`
2. Secure the AgentCore endpoint with IAS principal propagation (see the follow-up blog on securing with SAP Cloud Identity Services)
3. Extend `warehouseagentscenario.yaml` with input parameters (e.g. `product_id`) so Joule collects them before invoking the agent
4. Add multi-turn support by passing `context_id` from the Joule conversation to the agent